# 🏦 Credit Assessment Environment - GRPO Training

This notebook trains an LLM to make accurate loan underwriting decisions using **Group Relative Policy Optimization (GRPO)** from HuggingFace TRL.

## What the Agent Learns
- Follow RBI guidelines (CIBIL score, FOIR, LTV ratios)
- Detect **trap cases** (perfect profile with one hidden flaw)
- Make correct decisions: `approve`, `reject`, `request_docs`, `counter_offer`

## Theme Alignment
- **Theme #3: World Modeling** - Real professional banking task
- **Theme #4: Self-Improvement** - Curriculum learning + Adversarial self-play

## Training Modes (Choose ONE path)

| Mode | Section | Time | Best For |
|------|---------|------|----------|
| Standard | 7 only | ~45 min | Quick baseline |
| Curriculum | 7b only | ~60-90 min | **Recommended** |
| Full Pipeline | 7b → 7c | ~2-3 hours | Maximum improvement |

### Recommended Path: Curriculum + Adversarial
1. Run Section 7b (Curriculum Learning)
2. Then run Section 7c (Adversarial Self-Play)

This creates a **two-stage self-improvement loop** that aligns with Theme #4.

---

**Runtime:** T4 GPU (Colab Free Tier)  
**Training Time:** See table above  
**Expected Improvement:** 45% → 75%+ accuracy

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q trl>=0.17.0 transformers>=4.50.0 datasets accelerate peft bitsandbytes
!pip install -q matplotlib pydantic

In [ ]:
# Clone the Credit Assessment Environment
!git clone https://huggingface.co/spaces/iamnijin/credit-assessment-env
%cd credit-assessment-env

In [ ]:
# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Environment Setup

The Credit Assessment Environment simulates Indian loan underwriting with:
- **3 difficulty levels**: Personal (Easy), Vehicle (Medium), Home (Hard)
- **Trap cases**: Profiles designed to trick pattern-matching LLMs
- **Asymmetric rewards**: Approving bad loans costs 3× more than rejecting good ones

In [ ]:
# Import standalone training utilities (avoids complex import chains)
import json
import random
from typing import Any

from train_utils import (
    DIFFICULTY_PROFILES,
    generate_applicant,
    calculate_ground_truth,
    calculate_reward,
    build_profile_text,
    CreditAssessmentAction,
    LoanDecision,
    generate_adversarial_case,
    AdversarialTracker,
    ADVERSARIAL_STRATEGIES,
)

# Test the environment
applicant = generate_applicant(task_id=1)  # Personal Loan
ground_truth = calculate_ground_truth(applicant)
profile = build_profile_text(applicant)

print("=" * 60)
print("Sample Loan Application")
print("=" * 60)
print(profile)
print(f"\nGround Truth Decision: {ground_truth}")

## 3. Dataset Generation

We generate synthetic loan applications with known ground truth decisions.
The dataset includes good, bad, borderline, and **trap** profiles.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a senior loan officer at an Indian bank. You assess loan applications following RBI guidelines.

Respond with JSON containing:
- "decision": one of "approve", "reject", "request_docs", "counter_offer"
- "reasoning": brief explanation
- "counter_offer_amount": (only for counter_offer) reduced loan amount
- "docs_requested": (only for request_docs) needed documents

Key rules:
- CIBIL < 700 → reject
- FOIR > 50% → reject
- Missing documents → request_docs
- Home loan without RERA → reject (even if financials are perfect!)
- Vehicle loan LTV > 85% → counter_offer
- Home loan LTV: ≤30L→90%, 30-75L→80%, >75L→75%

Check EVERY criterion. One failure = rejection.

Respond with valid JSON only."""

def generate_dataset(num_samples: int, seed: int = 42, difficulty: str = "all") -> Dataset:
    """Generate loan application dataset with curriculum support."""
    random.seed(seed)
    samples = []
    
    for i in range(num_samples):
        task_id = (i % 3) + 1  # Balanced tasks
        applicant = generate_applicant(task_id, difficulty=difficulty)
        ground_truth = calculate_ground_truth(applicant)
        profile_text = build_profile_text(applicant)
        
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": profile_text}
        ]
        
        samples.append({
            "prompt": prompt,
            "ground_truth": ground_truth,
            "task_id": task_id,
            "applicant_data": json.dumps(applicant),
        })
    
    return Dataset.from_list(samples)

# Generate datasets
train_dataset = generate_dataset(300, seed=42)  # 300 for Colab
eval_dataset = generate_dataset(30, seed=123)

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

# Show distribution
from collections import Counter
gt_dist = Counter([s["ground_truth"] for s in train_dataset])
print(f"\nGround truth distribution: {dict(gt_dist)}")

## 4. Reward Function

The reward function evaluates model decisions against ground truth:

| Scenario | Reward |
|----------|--------|
| Correct decision | +10 |
| Request docs when incomplete | +2 |
| Counter-offer catches high LTV | +5 |
| Approve bad loan | -15 |
| Reject good loan | -5 |
| Approve non-RERA home | -20 |

In [ ]:
def credit_assessment_reward(
    completions: list,
    ground_truth: list[str],
    applicant_data: list[str],
    **kwargs
) -> list[float]:
    """Evaluate loan decisions against ground truth."""
    rewards = []
    
    for completion, gt, applicant_json in zip(completions, ground_truth, applicant_data):
        try:
            # Extract content
            if isinstance(completion, list) and len(completion) > 0:
                content = completion[0].get("content", "")
            else:
                content = str(completion)
            
            # Clean markdown
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]
            
            parsed = json.loads(content.strip())
            decision_str = parsed.get("decision", "reject").lower()
            
            try:
                decision = LoanDecision(decision_str)
            except ValueError:
                decision = LoanDecision.REJECT
            
            action = CreditAssessmentAction(
                decision=decision,
                reasoning=parsed.get("reasoning", ""),
                counter_offer_amount=parsed.get("counter_offer_amount"),
                docs_requested=parsed.get("docs_requested"),
            )
            
            applicant = json.loads(applicant_json)
            raw_reward = calculate_reward(action, applicant, gt)
            
            # Normalize [-20, +10] → [-1, +1]
            normalized = (raw_reward - (-20.0)) / (10.0 - (-20.0)) * 2 - 1
            rewards.append(normalized)
            
        except (json.JSONDecodeError, KeyError, TypeError):
            rewards.append(-0.5)  # Invalid JSON penalty
        except Exception:
            rewards.append(-0.3)
    
    return rewards

def format_reward(completions: list, **kwargs) -> list[float]:
    """Reward proper JSON formatting."""
    rewards = []
    
    for completion in completions:
        try:
            if isinstance(completion, list) and len(completion) > 0:
                content = completion[0].get("content", "")
            else:
                content = str(completion)
            
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]
            
            parsed = json.loads(content.strip())
            
            if "decision" in parsed and "reasoning" in parsed:
                rewards.append(0.2)
            elif "decision" in parsed:
                rewards.append(0.1)
            else:
                rewards.append(-0.1)
        except:
            rewards.append(-0.2)
    
    return rewards

def combined_reward(completions, ground_truth, applicant_data, **kwargs):
    """Combined reward: 80% decision accuracy + 20% format."""
    decision_rewards = credit_assessment_reward(completions, ground_truth, applicant_data, **kwargs)
    format_rewards = format_reward(completions, **kwargs)
    return [0.8 * d + 0.2 * f for d, f in zip(decision_rewards, format_rewards)]

print("✓ Reward functions defined")

## 5. Model & Training Configuration

We use:
- **Qwen2.5-1.5B-Instruct**: Small, fast, good at instruction following
- **LoRA**: Parameter-efficient fine-tuning (trains only 0.1% of parameters)
- **GRPO**: Group Relative Policy Optimization for reward maximization

In [ ]:
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# GRPO training config
training_args = GRPOConfig(
    output_dir="./grpo_credit_assessment",
    
    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    
    # GRPO specific
    num_generations=4,  # Completions per prompt
    max_completion_length=256,
    
    # Memory optimization
    gradient_checkpointing=True,
    bf16=True,
    
    # Logging
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    
    report_to="none",  # Change to "wandb" if you have W&B
)

print(f"Model: {MODEL_NAME}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"GRPO generations per prompt: {training_args.num_generations}")

In [ ]:
# Create GRPO Trainer
trainer = GRPOTrainer(
    model=MODEL_NAME,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
    reward_funcs=combined_reward,  # Combined: 80% decision + 20% format
)

print("✓ Trainer created successfully!")

## 6. Pre-Training Evaluation (Baseline)

Let's measure the model's accuracy **before** training.

In [ ]:
def evaluate_model(model, tokenizer, dataset, num_samples=20):
    """Evaluate model accuracy on loan decisions."""
    correct = 0
    total = 0
    results = []
    
    for i, sample in enumerate(dataset):
        if i >= num_samples:
            break
        
        prompt = tokenizer.apply_chat_template(
            sample["prompt"],
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.pad_token_id,
            )
        
        response = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )
        
        try:
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0]
            elif "```" in response:
                response = response.split("```")[1].split("```")[0]
            
            parsed = json.loads(response.strip())
            decision = parsed.get("decision", "").lower()
            is_correct = (decision == sample["ground_truth"])
            
            results.append({
                "task_id": sample["task_id"],
                "ground_truth": sample["ground_truth"],
                "predicted": decision,
                "correct": is_correct,
            })
            
            if is_correct:
                correct += 1
            total += 1
            
        except:
            results.append({
                "task_id": sample["task_id"],
                "ground_truth": sample["ground_truth"],
                "predicted": "[PARSE_ERROR]",
                "correct": False,
            })
            total += 1
    
    accuracy = correct / total if total > 0 else 0
    return accuracy, results

# Baseline evaluation
print("Evaluating pre-training baseline...")
baseline_acc, baseline_results = evaluate_model(
    trainer.model, tokenizer, eval_dataset, num_samples=20
)
print(f"\n📊 Baseline Accuracy: {baseline_acc*100:.1f}%")

## 7. Training! 🚀

Now we train the model with GRPO. You'll see:
- **reward/mean**: Average reward (should increase)
- **loss**: Training loss (should decrease)
- **clip_ratio**: GRPO clipping ratio

In [ ]:
# Train!
print("Starting GRPO training...")
print("This will take ~30-60 minutes on a T4 GPU.")
print("="*60)

trainer.train()

print("="*60)
print("✓ Training complete!")

In [ ]:
# Save the model
trainer.save_model("./grpo_credit_assessment_final")
print("✓ Model saved to ./grpo_credit_assessment_final")

## 7b. Curriculum Learning (Recommended)

**Run this INSTEAD of Section 7 for better results.**

Curriculum learning trains in 3 phases:
1. **Phase 1 (Easy)**: Only obvious good/bad cases
2. **Phase 2 (Medium)**: Adds borderline cases  
3. **Phase 3 (Hard)**: Adds all trap cases

This aligns with **Theme #4: Self-Improvement** and is now the **default** training mode.

In [ ]:
# CURRICULUM LEARNING - Run this INSTEAD of Section 7 for better results
# This is now the recommended training mode (Theme #4: Self-Improvement)

phases = [
    ("easy", "Phase 1: Learning Basics"),
    ("medium", "Phase 2: Refining"),
    ("hard", "Phase 3: Mastering Traps"),
]

phase_results = []

for phase_idx, (difficulty, phase_name) in enumerate(phases):
    print(f"\n{'='*60}")
    print(f"{phase_name}")
    print(f"{'='*60}")
    
    # Generate phase-specific dataset
    phase_train = generate_dataset(100, seed=42+phase_idx, difficulty=difficulty)
    
    # Update trainer dataset
    trainer.train_dataset = phase_train
    
    # Train this phase
    trainer.train()
    
    # Quick eval
    acc, _ = evaluate_model(trainer.model, tokenizer, eval_dataset, num_samples=10)
    phase_results.append((phase_name, acc))
    print(f"Accuracy after {phase_name}: {acc*100:.1f}%")

print("\n" + "="*60)
print("CURRICULUM RESULTS")
print("="*60)
for name, acc in phase_results:
    print(f"{name}: {acc*100:.1f}%")

# Save after curriculum training
trainer.save_model("./grpo_curriculum_model")
print("\n✓ Curriculum-trained model saved!")

## 7c. Adversarial Self-Play Training (Run After Curriculum)

**Run this AFTER Section 7b (Curriculum) for maximum improvement.**

Adversarial self-play is the key differentiator for **Theme #4: Self-Improvement**:
1. Evaluate model on adversarial cases (edge cases designed to trick LLMs)
2. Identify which strategies the model fails at most
3. Generate targeted training data focusing on weaknesses
4. Train on hard cases
5. Repeat

This creates a **self-improvement loop** where training dynamically adapts to the model's weaknesses.

In [ ]:
# ADVERSARIAL SELF-PLAY TRAINING
# Run this AFTER Section 7b (Curriculum) for the full self-improvement pipeline

from datasets import Dataset

def generate_adversarial_dataset(num_samples, seed=42, tracker=None, target_weakness=True):
    """Generate adversarial cases targeting model weaknesses."""
    random.seed(seed)
    samples = []
    
    if tracker and target_weakness:
        cases = tracker.generate_targeted_batch(num_samples, target_weakness=True)
        for case in cases:
            applicant = case["applicant"]
            strategy = case["strategy"]
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            
            prompt = [{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": profile_text}]
            
            samples.append({
                "prompt": prompt,
                "ground_truth": ground_truth,
                "task_id": {"personal": 1, "vehicle": 2, "home": 3}[applicant["loan_type"]],
                "applicant_data": json.dumps(applicant),
                "strategy": strategy,
            })
    else:
        for i in range(num_samples):
            strategy = random.choice(ADVERSARIAL_STRATEGIES)
            applicant = generate_adversarial_case(strategy)
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            
            prompt = [{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": profile_text}]
            
            samples.append({
                "prompt": prompt,
                "ground_truth": ground_truth,
                "task_id": {"personal": 1, "vehicle": 2, "home": 3}[applicant["loan_type"]],
                "applicant_data": json.dumps(applicant),
                "strategy": strategy,
            })
    
    return Dataset.from_list(samples)

def evaluate_adversarial_strategies(model, tokenizer, num_per_strategy=2):
    """Evaluate model on each adversarial strategy."""
    results = {s: {"correct": 0, "total": 0} for s in ADVERSARIAL_STRATEGIES}
    
    for strategy in ADVERSARIAL_STRATEGIES:
        for _ in range(num_per_strategy):
            applicant = generate_adversarial_case(strategy)
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            
            prompt = [{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": profile_text}]
            
            formatted = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
            
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                                         temperature=0.7, pad_token_id=tokenizer.pad_token_id)
            
            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            
            try:
                if "```json" in response:
                    response = response.split("```json")[1].split("```")[0]
                elif "```" in response:
                    response = response.split("```")[1].split("```")[0]
                
                parsed = json.loads(response.strip())
                decision = parsed.get("decision", "").lower()
                
                results[strategy]["total"] += 1
                if decision == ground_truth:
                    results[strategy]["correct"] += 1
            except:
                results[strategy]["total"] += 1
    
    return results

# Initialize tracker
tracker = AdversarialTracker()
adversarial_rounds = 3
samples_per_round = 50

print("=" * 60)
print("ADVERSARIAL SELF-PLAY TRAINING")
print("=" * 60)
print(f"Rounds: {adversarial_rounds}")
print(f"Samples per round: {samples_per_round}")

round_results = []

for round_idx in range(adversarial_rounds):
    print(f"\n{'='*60}")
    print(f"ROUND {round_idx + 1}/{adversarial_rounds}")
    print(f"{'='*60}")
    
    # Step 1: Evaluate on adversarial cases
    print("Evaluating on adversarial strategies...")
    eval_results = evaluate_adversarial_strategies(trainer.model, tokenizer, num_per_strategy=2)
    
    # Update tracker
    for strategy, results in eval_results.items():
        for _ in range(results["correct"]):
            tracker.record_result(strategy, True)
        for _ in range(results["total"] - results["correct"]):
            tracker.record_result(strategy, False)
    
    # Print current weaknesses
    print("\nStrategy accuracy:")
    for strategy, results in sorted(eval_results.items(), key=lambda x: x[1]["correct"]/max(1, x[1]["total"])):
        acc = results["correct"] / max(1, results["total"]) * 100
        print(f"  {strategy}: {acc:.0f}%")
    
    # Identify weakness
    weakness = tracker.get_weakness()
    print(f"\n→ Targeting weakness: {weakness}")
    
    # Step 2: Generate targeted dataset
    adversarial_data = generate_adversarial_dataset(samples_per_round, seed=42+round_idx+100, 
                                                     tracker=tracker, target_weakness=True)
    
    # Mix with some regular hard cases
    regular_hard = generate_dataset(samples_per_round // 2, seed=42+round_idx+200, difficulty="hard")
    combined = list(adversarial_data) + list(regular_hard)
    random.shuffle(combined)
    train_data = Dataset.from_list(combined)
    
    # Step 3: Train
    trainer.train_dataset = train_data
    print(f"\nTraining on {len(train_data)} samples...")
    trainer.train()
    
    # Calculate round accuracy
    total_correct = sum(r["correct"] for r in eval_results.values())
    total = sum(r["total"] for r in eval_results.values())
    round_acc = total_correct / total if total > 0 else 0
    round_results.append((round_idx + 1, weakness, round_acc))
    
    print(f"\nRound {round_idx + 1} complete: {round_acc*100:.1f}% accuracy")

# Summary
print("\n" + "=" * 60)
print("ADVERSARIAL TRAINING RESULTS")
print("=" * 60)
for round_num, weakness, acc in round_results:
    print(f"Round {round_num}: {acc*100:.1f}% (targeted: {weakness})")

print("\nFinal weakness analysis:")
summary = tracker.get_summary()
for strategy, data in sorted(summary.items(), key=lambda x: x[1]["accuracy"]):
    print(f"  {strategy}: {data['accuracy']*100:.0f}% ({data['failures']} failures)")

# Save final model
trainer.save_model("./grpo_adversarial_final")
print("\n✓ Adversarial-trained model saved!")

## 8. Post-Training Evaluation

Let's measure improvement!

In [ ]:
# Post-training evaluation
print("Evaluating trained model...")
trained_acc, trained_results = evaluate_model(
    trainer.model, tokenizer, eval_dataset, num_samples=20
)

print(f"\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"📊 Baseline Accuracy: {baseline_acc*100:.1f}%")
print(f"📈 Trained Accuracy:  {trained_acc*100:.1f}%")
print(f"🚀 Improvement:       {(trained_acc - baseline_acc)*100:+.1f}%")
print("="*60)

In [ ]:
# Detailed breakdown by task
import pandas as pd

task_names = {1: "Personal Loan", 2: "Vehicle Loan", 3: "Home Loan"}

# Baseline breakdown
baseline_df = pd.DataFrame(baseline_results)
baseline_by_task = baseline_df.groupby("task_id")["correct"].mean()

# Trained breakdown
trained_df = pd.DataFrame(trained_results)
trained_by_task = trained_df.groupby("task_id")["correct"].mean()

print("\nAccuracy by Loan Type:")
print("-" * 50)
print(f"{'Task':<20} {'Baseline':>12} {'Trained':>12} {'Δ':>10}")
print("-" * 50)
for task_id in [1, 2, 3]:
    base = baseline_by_task.get(task_id, 0)
    trained = trained_by_task.get(task_id, 0)
    delta = trained - base
    print(f"{task_names[task_id]:<20} {base*100:>10.1f}% {trained*100:>10.1f}% {delta*100:>+9.1f}%")

## 9b. Generate Hackathon-Ready Results & Narrative

Run this cell after training to generate:
1. **Before/After accuracy chart** (for your presentation)
2. **Trap case examples** (for your pitch)
3. **Copy-paste ready pitch summary**

In [ ]:
# ============================================================
# HACKATHON RESULTS GENERATOR
# ============================================================
# Uses your actual baseline_acc and trained_acc from above

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ------------------------------------------------------------
# 1. GENERATE RESULTS CHART
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overall accuracy comparison
categories = ['Baseline\n(Pre-training)', 'Trained\n(Post-GRPO)']
accuracies = [baseline_acc * 100, trained_acc * 100]
colors = ['#e74c3c', '#27ae60']

bars = axes[0].bar(categories, accuracies, color=colors, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Overall Loan Decision Accuracy', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].axhline(y=100, color='gray', linestyle='--', alpha=0.3, label='Perfect (Rule-Based)')

# Add value labels
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=14)

# Add improvement arrow
improvement = (trained_acc - baseline_acc) * 100
mid_y = (baseline_acc + trained_acc) / 2 * 100
axes[0].annotate(f'+{improvement:.1f}%', 
                 xy=(1, trained_acc * 100 - 5), 
                 fontsize=16, fontweight='bold', color='#27ae60',
                 ha='center')

# Right: By loan type breakdown
task_names = ['Personal\n(Easy)', 'Vehicle\n(Medium)', 'Home\n(Hard)']

# Calculate per-task accuracy from results
baseline_by_task = [0, 0, 0]
trained_by_task = [0, 0, 0]
baseline_counts = [0, 0, 0]
trained_counts = [0, 0, 0]

for r in baseline_results:
    idx = r['task_id'] - 1
    baseline_by_task[idx] += 1 if r['correct'] else 0
    baseline_counts[idx] += 1

for r in trained_results:
    idx = r['task_id'] - 1
    trained_by_task[idx] += 1 if r['correct'] else 0
    trained_counts[idx] += 1

baseline_by_task = [baseline_by_task[i]/max(1,baseline_counts[i])*100 for i in range(3)]
trained_by_task = [trained_by_task[i]/max(1,trained_counts[i])*100 for i in range(3)]

x = range(len(task_names))
width = 0.35

bars1 = axes[1].bar([i - width/2 for i in x], baseline_by_task, width, 
                    label='Baseline', color='#e74c3c', edgecolor='black')
bars2 = axes[1].bar([i + width/2 for i in x], trained_by_task, width,
                    label='Trained', color='#27ae60', edgecolor='black')

axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy by Loan Type (Difficulty)', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(task_names)
axes[1].set_ylim(0, 100)
axes[1].legend()

# Add value labels
for bar in bars1:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('hackathon_results.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n✓ Chart saved to hackathon_results.png")
print("  (Right-click → Save Image to download)")

# ------------------------------------------------------------
# 2. TRAP CASE EXAMPLES (for your pitch)
# ------------------------------------------------------------

print("\n" + "="*60)
print("TRAP CASES FOR YOUR PITCH")
print("="*60)

# Generate example trap cases
trap_examples = [
    {
        "name": "The High-Income Trap",
        "profile": "₹6L/month income, 20% FOIR, 12 years experience",
        "hidden_flaw": "CIBIL score: 699 (just 1 point below 700)",
        "correct": "REJECT",
        "why_fails": "LLMs see high income and approve without checking CIBIL threshold"
    },
    {
        "name": "The Perfect-Profile Trap", 
        "profile": "CIBIL 820, ₹2L/month, 25% FOIR, 10 years",
        "hidden_flaw": "Property NOT RERA registered",
        "correct": "REJECT",
        "why_fails": "Everything screams 'approve' - RERA field is easy to overlook"
    },
    {
        "name": "The RBI Tier Trap",
        "profile": "₹1.2Cr home loan, LTV 78%",
        "hidden_flaw": "Loans >₹75L have max 75% LTV (RBI rule)",
        "correct": "COUNTER_OFFER",
        "why_fails": "Must know tiered limits: ≤30L→90%, 30-75L→80%, >75L→75%"
    }
]

for i, trap in enumerate(trap_examples, 1):
    print(f"\n### Trap {i}: {trap['name']}")
    print(f"Profile: {trap['profile']}")
    print(f"Hidden Flaw: {trap['hidden_flaw']}")
    print(f"Correct Decision: {trap['correct']}")
    print(f"Why LLMs Fail: {trap['why_fails']}")

# ------------------------------------------------------------
# 3. COPY-PASTE PITCH SUMMARY
# ------------------------------------------------------------

print("\n" + "="*60)
print("COPY-PASTE PITCH SUMMARY")
print("="*60)

print(f"""
## Credit Assessment Environment: Teaching LLMs to Be Loan Officers

**The Problem:** LLMs pattern-match. Banking requires precision. 
One overlooked criterion (CIBIL 699 vs 700) = ₹50L NPA.

**Our Solution:** Self-improving RL environment with:
- Trap cases targeting LLM weaknesses
- Curriculum learning (easy → medium → hard)  
- Adversarial self-play based on failure analysis
- Asymmetric rewards matching real banking risk

**Results:**
- Baseline: {baseline_acc*100:.1f}%
- Trained: {trained_acc*100:.1f}%
- Improvement: +{improvement:.1f}%

**Why It Matters:**
Every bank in India processes thousands of loans daily. 
An agent that catches edge cases = millions saved in NPAs.
""")

## 9. Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

# Get training history
history = trainer.state.log_history

# Extract metrics
steps = []
rewards = []
losses = []

for entry in history:
    if "reward" in entry:
        steps.append(entry.get("step", 0))
        rewards.append(entry["reward"])
    if "loss" in entry and entry.get("step", 0) > 0:
        if len(losses) < len(steps):
            losses.append(entry["loss"])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward curve
axes[0].plot(steps[:len(rewards)], rewards, 'b-', linewidth=2, label='Average Reward')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Reward')
axes[0].set_title('🎯 Reward During Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
categories = ['Baseline', 'Trained']
accuracies = [baseline_acc * 100, trained_acc * 100]
colors = ['#ff6b6b', '#51cf66']
bars = axes[1].bar(categories, accuracies, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('📈 Before vs After Training')
axes[1].set_ylim(0, 100)

# Add value labels
for bar, acc in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved to training_results.png")

## 10. Try the Trained Model!

Let's test on a trap case to see if training helped.

In [ ]:
# Generate a trap case
random.seed(12345)
trap_applicant = generate_applicant(task_id=3)  # Home loan (hardest)
trap_gt = calculate_ground_truth(trap_applicant)
trap_profile = build_profile_text(trap_applicant)

print("=" * 60)
print("TEST CASE: Home Loan Application")
print("=" * 60)
print(trap_profile)
print(f"\n🎯 Correct Decision: {trap_gt}")
print("=" * 60)

# Get model's decision
test_prompt = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": trap_profile}
]

formatted = tokenizer.apply_chat_template(
    test_prompt, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(formatted, return_tensors="pt").to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("\n🤖 Model's Response:")
print(response)

# Check if correct
try:
    if "```json" in response:
        response = response.split("```json")[1].split("```")[0]
    elif "```" in response:
        response = response.split("```")[1].split("```")[0]
    
    parsed = json.loads(response.strip())
    decision = parsed.get("decision", "")
    
    if decision == trap_gt:
        print("\n✅ CORRECT!")
    else:
        print(f"\n❌ Incorrect. Model said '{decision}', should be '{trap_gt}'")
except:
    print("\n⚠️ Could not parse response")

## 11. Push to HuggingFace Hub (Optional)

Upload your trained model to share with others!

In [ ]:
# Uncomment and run to push to hub
# from huggingface_hub import login
# login()  # Enter your HF token

# trainer.push_to_hub("your-username/credit-assessment-grpo")
# print("✓ Model pushed to HuggingFace Hub!")

---

## Summary

This notebook demonstrated:

1. **Environment**: Credit Assessment with 3 difficulty levels and trap cases
2. **Reward Model**: Asymmetric penalties matching real banking risk
3. **Training**: GRPO with LoRA for parameter-efficient fine-tuning
4. **Results**: Measurable improvement in loan decision accuracy

### Training Paths (all code is ready to run):

| Path | What to Run | Description |
|------|-------------|-------------|
| Quick | Section 7 | Standard training, all difficulties mixed |
| **Recommended** | Section 7b | Curriculum: Easy → Medium → Hard |
| Full Pipeline | 7b then 7c | Curriculum + Adversarial self-play |